# Movie Recommendation System Using Content-Based Filtering
## Project Overview

Recommendation systems are machine learning applications designed to suggest relevant items to users based on available information.

This project develops a content-based movie recommendation system using movie metadata such as genres, cast, crew, keywords, and movie overviews. The system recommends movies that are similar to a selected movie by comparing their features using vectorization and cosine similarity.
## Objectives

### Main Objective

To build a content-based movie recommendation system using movie metadata.

### Specific Objectives

- Load and explore the movie dataset.
- Clean and preprocess movie metadata.
- Combine relevant movie features.
- Convert textual data into numerical vectors.
- Compute similarities between movies.
- Recommend the most similar movies.

## Recommendation Approach

This project uses a **Content-Based Recommendation System** because the dataset contains movie metadata such as cast, crew, genres, keywords, and descriptions but does not contain user ratings or viewing history.

Movies are represented using their textual features, which are transformed into vectors. Cosine Similarity is then used to identify movies that are most similar to one another.

## Machine Learning Pipeline

1. Import libraries
2. Load datasets
3. Explore the data
4. Data cleaning
5. Feature engineering
6. Text preprocessing
7. Vectorization (CountVectorizer or TF-IDF)
8. Compute cosine similarity
9. Build recommendation function
10. Evaluate recommendations

In [1]:
# Imporatation of the libraries 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot
import difflib

# similarity measure 
from sklearn.metrics.pairwise import cosine_similarity 
# Text vectorization 
from sklearn.feature_extraction.text import TfidfVectorizer
from difflib import get_close_matches



In [2]:
credits=pd.read_csv('credits.csv')
movies = pd.read_csv('movies_metadata.csv')
credits.head(5)
movies.head(5)



/tmp/ipykernel_22963/3944592089.py:2: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
#checking the duplicates 
credits.duplicated().sum()
credits.drop_duplicates(inplace=True)
credits.duplicated().sum()
movies.duplicated().sum()
movies.drop_duplicates(inplace=True)

In [4]:
#checking the null values 
credits.isnull().sum()
movies.isnull().sum()

adult                        0
belongs_to_collection    40959
budget                       0
genres                       0
homepage                 37673
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25045
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [5]:
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

In [6]:
#merging the data sets 
data = credits.merge(movies , on= 'id')

In [7]:
text_columns = [
    'overview',
    'genres',
    'cast',
    'crew',
    'original_language',
    'title'
]

data[text_columns] = data[text_columns].fillna('')

In [8]:
data.columns

Index(['cast', 'crew', 'id', 'adult', 'belongs_to_collection', 'budget',
       'genres', 'homepage', 'imdb_id', 'original_language', 'original_title',
       'overview', 'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [37]:
data["tags"] = (data['cast']+" "+ data['overview']+' ' + data['crew']+ ' '+  
                 ' ' + data['genres'] + " " + data['title']+ ' '+ data['original_language'])
data['tags']=data['tags'].str.lower()

In [16]:
#Vectorization 
tfidf = TfidfVectorizer(stop_words='english',
                        max_features=5000,# Find the movie index,
    )
vectors = tfidf.fit_transform(data['tags'])

In [ ]:
def recommend(movie , n=5):
    movie_list = data["title"].dropna().tolist()

    #check if the movie exists 
    if movie not in movie_list:
        suggestion = get_close_matches(movie , movie_list , n=5 , cutoff=0.4)
        if suggestion:
            movie= suggestion[0]
            print('\n movie not found')
            print("\nDid you mean:")
            for s in suggestion:
                print(f"- {s}")
        else:
            print('\n movie not found and no similar titile found')
            return
    # Find the movie index
    index = data[data['title'] == movie].index[0]

    # Compute similarity with all movies
    similarity = cosine_similarity(vectors[index], vectors).flatten()
    movie_indices = similarity.argsort()[::-1][1:6]


    print(f"Top {n} recommendations for '{movie}':\n")

    for i in movie_indices:
        print(f"{data.iloc[i]['title']} ({similarity[i]:.3f})")
    
    

In [43]:
recommend("avenger")


 movie not found

Did you mean:
- Avenger
- Scavengers
- Lavender
- Passenger
- Revenge
Top 5 recommendations for 'Avenger':

Now You See Him, Now You Don't (0.408)
The Cure (0.358)
The November Man (0.339)
Dexter's Laboratory: Ego Trip (0.335)
The Man with One Red Shoe (0.326)
